<a href="https://colab.research.google.com/github/QuintonPang/Titanic-Machine-Learning-from-Disaster/blob/main/Titanic_Machine_Learning_from_Disaster.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd

In [2]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

In [3]:
test_passenger_ids = test_df["PassengerId"]

In [4]:
combined_df = pd.concat([train_df, test_df],axis=0).reset_index(drop=True)

In [5]:
# feature engineering : extract titles
combined_df["Title"] = combined_df["Name"].str.extract(r"([A-Za-z]+)\.")

In [6]:
title_mapping = {
    "Mr": "Mr",
    "Miss": "Miss",
    "Mrs": "Mrs",
    "Master": "Master",
    "Mlle": "Miss",
    "Ms": "Miss",
    "Mme": "Mrs",
}

In [7]:
combined_df["Title"] = combined_df["Title"].map(title_mapping).fillna("Rare")

In [8]:
# feature engineering : Family size
combined_df["FamilySize"] = combined_df["SibSp"] + combined_df["Parch"] + 1

In [9]:
# imputation

In [10]:
combined_df["Age"] = combined_df.groupby(["Pclass","Sex","Title"])["Age"].transform(lambda x:x.fillna(x.median()))

In [11]:
combined_df["Fare"] = combined_df.groupby("Pclass")["Fare"].transform(lambda x: x.fillna(x.median()))

In [12]:
combined_df["Embarked"] = combined_df["Embarked"].fillna(combined_df["Embarked"].mode()[0])

In [13]:
# one hot encoding
combined_df = pd.get_dummies(combined_df, columns=["Sex","Embarked","Title"], drop_first=True)


In [14]:
# drop unused features
features_to_drop = ["PassengerId","Name","Ticket","Cabin","Survived"]
X_features = combined_df.drop(columns=features_to_drop)

In [15]:
# separate back into train and test sets
X_train = X_features.iloc[:len(train_df)].copy()
y_train = train_df["Survived"].astype(int)
X_test = X_features.iloc[len(train_df):].copy()

In [16]:
# model training
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
model = RandomForestClassifier(n_estimators = 100, max_depth=5, random_state=42)

In [17]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [18]:
cv_scores = cross_val_score(model,X_train, y_train, cv= cv, scoring="accuracy")

In [19]:
print(f"Mean CV Accuracy: {cv_scores.mean() * 100:.2f}%")

Mean CV Accuracy: 83.28%


In [20]:
model.fit(X_train, y_train)
predictions = model.predict(X_test)

In [21]:
train_accuracy = model.score(X_train, y_train)
train_accuracy

0.8428731762065096

In [22]:
submission = pd.DataFrame({"PassengerId": test_passenger_ids, "Survived": predictions})
submission.to_csv("submission.csv", index=False)